In [5]:
from cresowlve.utils import read_json
import numpy as np

results = read_json("results.json")
reasoning_type_results = read_json("../experiments/data/task/chgk_benchmark_reasoning_types.json")
factual_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "factual"]
creative_ids = [s["id"] for s in reasoning_type_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "creative"]
latex_template = "\\textsc{{{model_name}}} & {thinking} & {en_em} & {en_llm} & {ru_em} & {ru_llm} \\\\"

non_thinking_outputs = []
thinking_outputs = []
non_thinking_em_diffs = []
non_thinking_llm_diffs = []
thinking_em_diffs = []
thinking_llm_diffs = []
json_results = []

for result in results:
    en_em, en_llm, ru_em, ru_llm = 0, 0, 0, 0
    en_outputs = read_json(result["en_path"]) if result["en_path"] else None
    ru_outputs = read_json(result["ru_path"]) if result["ru_path"] else None
    
    if en_outputs:
        en_outputs["data"] = [s for s in en_outputs["data"] if s["id"] in creative_ids]
        en_em = np.mean([int(s.get("exact_match", 0)) for s in en_outputs["data"]])
        en_llm = np.mean([int(s.get("gpt-4o_judge_match", 0)) for s in en_outputs["data"]])
    if ru_outputs:
        ru_outputs["data"] = [s for s in ru_outputs["data"] if s["id"] in creative_ids]
        ru_em = np.mean([int(s.get("exact_match", 0)) for s in ru_outputs["data"]])
        ru_llm = np.mean([int(s.get("gpt-4o_judge_match", 0)) for s in ru_outputs["data"]])

    output = latex_template.format(
        model_name=result["model_name"],
        thinking=result["thinking"] if result["thinking"] is not None else "-",
        en_em=f"{en_em*100:.2f}",
        en_llm=f"{en_llm*100:.2f}",
        ru_em=f"{ru_em*100:.2f}",
        ru_llm=f"{ru_llm*100:.2f}"
    )

    if result["thinking"] is None:
        non_thinking_em_diffs.append(en_em - ru_em)
        non_thinking_llm_diffs.append(en_llm - ru_llm)
        non_thinking_outputs.append(output)
    else:
        thinking_em_diffs.append(en_em - ru_em)
        thinking_llm_diffs.append(en_llm - ru_llm)
        thinking_outputs.append(output)

    json_results.append({
        "model_family": result["model_name"].split("-")[0],
        "model_name": result["model_name"],
        "thinking_model": "yes" if result["thinking"] is not None else "no",
        "thinking_effort": result["thinking"],
        "exact_match": en_em.item(),
        "llm_judge": en_llm.item(),
    })

print("Non-thinking models:")
for output in non_thinking_outputs:
    print(output)

print("\nThinking models:")
for output in thinking_outputs:
    print(output)

# Additional analysis: EM and LLM judge match differences between English and Russian outputs
print("\nThinking models EM differences (English - Russian):")
print([f"{diff*100:.2f}" for diff in thinking_em_diffs])
print("\nThinking models LLM Judge Match differences (English - Russian):")
print([f"{diff*100:.2f}" for diff in thinking_llm_diffs])

print("\nNon-thinking models EM differences (English - Russian):")
print([f"{diff*100:.2f}" for diff in non_thinking_em_diffs])
print("\nNon-thinking models LLM Judge Match differences (English - Russian):")
print([f"{diff*100:.2f}" for diff in non_thinking_llm_diffs])

Non-thinking models:
\textsc{OLMo-2-32B-Instruct} & - & 2.09 & 5.29 & 0.44 & 1.31 \\
\textsc{C4AI-command-a} & - & 4.46 & 9.07 & 3.74 & 7.08 \\
\textsc{Llama-3.3-70B-Instruct} & - & 5.05 & 10.14 & 3.64 & 7.04 \\
\textsc{Mistral-Large-3-675B-Instruct} & - & 11.26 & 17.61 & 15.48 & 22.22 \\
\textsc{Qwen3-235B-A22B-Instruct} & - & 13.05 & 20.82 & 13.54 & 20.14 \\
\textsc{GPT-4.1-mini} & - & 7.33 & 13.78 & 4.85 & 8.83 \\
\textsc{GPT-4.1} & - & 13.44 & 24.99 & 17.03 & 26.06 \\

Thinking models:
\textsc{Qwen3.5-397B-A17B} & adaptive & 19.36 & 31.25 & 26.15 & 35.71 \\
\textsc{Qwen3-235B-A22B-Thinking} & adaptive & 13.49 & 20.82 & 13.34 & 21.11 \\
\textsc{DeepSeek-V3.2} & adaptive & 15.33 & 25.47 & 10.48 & 26.44 \\
\textsc{GLM-5} & adaptive & 18.92 & 37.17 & 28.87 & 44.25 \\
\textsc{GPT-5.4} & none & 9.41 & 20.96 & 15.38 & 26.54 \\
\textsc{GPT-5.4} & medium & 17.81 & 53.47 & 26.88 & 68.22 \\
\textsc{Gemini-3-Flash} & minimal & 24.99 & 41.10 & 39.98 & 54.34 \\
\textsc{Gemini-3-Flash} & medium &

In [6]:
json_results

[{'model_family': 'OLMo',
  'model_name': 'OLMo-2-32B-Instruct',
  'thinking_model': 'no',
  'thinking_effort': None,
  'exact_match': 0.020863658418243572,
  'llm_judge': 0.052886948083454635},
 {'model_family': 'C4AI',
  'model_name': 'C4AI-command-a',
  'thinking_model': 'no',
  'thinking_effort': None,
  'exact_match': 0.044638524987869965,
  'llm_judge': 0.09073265405143134},
 {'model_family': 'Llama',
  'model_name': 'Llama-3.3-70B-Instruct',
  'thinking_model': 'no',
  'thinking_effort': None,
  'exact_match': 0.050460941290635615,
  'llm_judge': 0.10140708393983504},
 {'model_family': 'Mistral',
  'model_name': 'Mistral-Large-3-675B-Instruct',
  'thinking_model': 'no',
  'thinking_effort': None,
  'exact_match': 0.11256671518680253,
  'llm_judge': 0.17612809315866085},
 {'model_family': 'Qwen3',
  'model_name': 'Qwen3-235B-A22B-Instruct',
  'thinking_model': 'no',
  'thinking_effort': None,
  'exact_match': 0.13051916545366327,
  'llm_judge': 0.2081513828238719},
 {'model_famil

In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from cresowlve.utils import read_json

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
difficulties = ["Very Simple", "Simple", "Medium", "Hard", "Very Hard"]
# ─────────────────────────────────────────────────────────────────────────────
benchmark = read_json("../experiments/data/task/chgk_benchmark.json")
benchmark_dict = {item["id"]: item for item in benchmark["data"]}

def plot_diff_results_by_path(results, path="en_path", metrics=None):
    if metrics is None:
        metrics = [
            ("exact_match", "Exact Match Accuracy"),
            ("gpt-4o_judge_match", "GPT-4o Judge Accuracy"),
        ]

    # Earthy palette for model lines.
    earthy_palette = [
        "#7A5C3E",  # clay brown
        "#6B8E23",  # olive
        "#A97142",  # terracotta
        "#8B6F47",  # muted ochre
        "#5E7D6A",  # moss green
        "#B08968",  # sand
        "#4F6D5A",  # forest sage
        "#9C6644",  # cinnamon
        "#A3B18A",  # light olive
        "#6C584C",  # walnut
    ]

    # Bootstrap sample accuracy by difficulty for each model and each metric.
    metric_frames = {}
    for metric_name, metric_label in metrics:
        rows = []

        for result in results:
            model_name = result["model_name"]
            thinking = result.get("thinking")
            key = f"{model_name} ({thinking})" if thinking is not None else model_name
            outputs = read_json(result[path]) if result[path] else None

            for diff_idx, diff_label in enumerate(difficulties, start=1):
                if outputs is None:
                    continue

                diff_outputs = [
                    output for output in outputs["data"]
                    if benchmark_dict.get(output["id"])
                    and benchmark_dict[output["id"]]["difficulty_id"] == diff_idx
                    and output["id"] in creative_ids  # Filter to creative questions only
                ]

                if not diff_outputs:
                    continue

                for _ in range(1000):
                    sample = np.random.choice(diff_outputs, len(diff_outputs), replace=True)
                    acc = float(np.mean([int(s.get(metric_name, 0)) for s in sample]))
                    rows.append({"Difficulty": diff_label, "Model": key, "Accuracy": acc})

        df = pd.DataFrame(rows)
        if not df.empty:
            df["Difficulty"] = pd.Categorical(df["Difficulty"], categories=difficulties, ordered=True)
        metric_frames[metric_label] = df

    # Two horizontal subplots with one shared legend under both plots.
    fig, axes = plt.subplots(1, len(metrics), figsize=(18, 6), sharey=True, facecolor="#FAF8F3")
    if len(metrics) == 1:
        axes = [axes]

    markers = ["o", "s", "^", "D", "P", "X", "*", "v", "<", ">"]
    legend_handles = None
    legend_labels = None

    for ax, (_, metric_label) in zip(axes, metrics):
        df = metric_frames[metric_label]
        ax.set_facecolor("#FCFBF7")

        if df.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=12, color="#4D463F")
            ax.set_title(metric_label, fontsize=18, color="#3E3A36")
            continue

        n_models = df["Model"].nunique()
        point = sns.pointplot(
            data=df,
            x="Difficulty",
            y="Accuracy",
            hue="Model",
            dodge=0.4,
            markers=markers[:n_models],
            linestyles="-",
            errorbar="sd",
            capsize=0.05,
            markersize=8,
            linewidth=2.5,
            err_kws={"linewidth": 1.5},
            palette=earthy_palette,
            ax=ax,
        )

        if legend_handles is None:
            legend_handles, legend_labels = point.get_legend_handles_labels()

        # Remove per-axis legends; we will draw one global legend below.
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        ax.grid(True, axis="y", color="#D9D3C7", linestyle="-", linewidth=0.8, alpha=0.8)
        ax.grid(True, axis="x", color="#E8E3DA", linestyle=":", linewidth=0.7, alpha=0.6)

        ax.set_title(metric_label, fontsize=18, color="#3E3A36")
        ax.set_xlabel("Difficulty", fontsize=16, color="#4D463F")
        ax.set_ylabel("Accuracy", fontsize=16, color="#4D463F")
        ax.tick_params(labelsize=13, colors="#3E3A36")
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines["left"].set_color("#B8AD98")
        ax.spines["bottom"].set_color("#B8AD98")
        ax.set_ylim(0, 1)

    fig.legend(
        legend_handles,
        legend_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.06),
        ncol=4,
        frameon=True,
        framealpha=0.95,
        facecolor="#F3EFE6",
        edgecolor="#CFC6B4",
        fontsize=14,
        title="Model",
        title_fontsize=18,
    )

    plt.tight_layout(rect=[0, 0.12, 1, 1])
    plt.savefig(f"../figures/diff-results-{path}.pdf", dpi=360, bbox_inches="tight", facecolor="white")
    plt.close()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from cresowlve.utils import read_json

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
difficulties = ["Very Simple", "Simple", "Medium", "Hard", "Very Hard"]
# ─────────────────────────────────────────────────────────────────────────────
benchmark = read_json("../experiments/data/task/chgk_benchmark.json")
benchmark_dict = {item["id"]: item for item in benchmark["data"]}

def plot_diff_results_by_metric(results, metric="exact_match"):
    paths = [
        ("en_path", "CresOWLve-En"),
        ("ru_path", "CresOWLve-Ru"),
    ]

    # Earthy palette for model lines.
    earthy_palette = [
        "#7A5C3E",  # clay brown
        "#6B8E23",  # olive
        "#A97142",  # terracotta
        "#8B6F47",  # muted ochre
        "#5E7D6A",  # moss green
        "#B08968",  # sand
        "#4F6D5A",  # forest sage
        "#9C6644",  # cinnamon
        "#A3B18A",  # light olive
        "#6C584C",  # walnut
    ]

    # Bootstrap sample accuracy by difficulty for each model and each path.
    path_frames = {}
    for path, path_label in paths:
        rows = []

        for result in results:
            model_name = result["model_name"]
            thinking = result.get("thinking")
            key = f"{model_name} ({thinking})" if thinking is not None else model_name
            outputs = read_json(result[path]) if result[path] else None

            for diff_idx, diff_label in enumerate(difficulties, start=1):
                if outputs is None:
                    continue

                diff_outputs = [
                    output for output in outputs["data"]
                    if benchmark_dict.get(output["id"])
                    and benchmark_dict[output["id"]]["difficulty_id"] == diff_idx
                    and output["id"] in creative_ids  # Filter to creative questions only
                ]

                if not diff_outputs:
                    continue

                for _ in range(1000):
                    sample = np.random.choice(diff_outputs, len(diff_outputs), replace=True)
                    acc = float(np.mean([int(s.get(metric, 0)) for s in sample]))
                    rows.append({"Difficulty": diff_label, "Model": key, "Accuracy": acc})

        df = pd.DataFrame(rows)
        if not df.empty:
            df["Difficulty"] = pd.Categorical(df["Difficulty"], categories=difficulties, ordered=True)
        path_frames[path_label] = df

    # Two horizontal subplots with one shared legend under both plots.
    fig, axes = plt.subplots(1, len(paths), figsize=(18, 6), sharey=True, facecolor="#FAF8F3")
    if len(paths) == 1:
        axes = [axes]

    markers = ["o", "s", "^", "D", "P", "X", "*", "v", "<", ">"]
    legend_handles = None
    legend_labels = None

    for ax, (_, path_label) in zip(axes, paths):
        df = path_frames[path_label]
        ax.set_facecolor("#FCFBF7")

        if df.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=12, color="#4D463F")
            ax.set_title(path_label, fontsize=18, color="#3E3A36")
            continue

        n_models = df["Model"].nunique()
        point = sns.pointplot(
            data=df,
            x="Difficulty",
            y="Accuracy",
            hue="Model",
            dodge=0.4,
            markers=markers[:n_models],
            linestyles="-",
            errorbar="sd",
            capsize=0.05,
            markersize=8,
            linewidth=2.5,
            err_kws={"linewidth": 1.5},
            palette=earthy_palette,
            ax=ax,
        )

        if legend_handles is None:
            legend_handles, legend_labels = point.get_legend_handles_labels()

        # Remove per-axis legends; we will draw one global legend below.
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        ax.grid(True, axis="y", color="#D9D3C7", linestyle="-", linewidth=0.8, alpha=0.8)
        ax.grid(True, axis="x", color="#E8E3DA", linestyle=":", linewidth=0.7, alpha=0.6)

        ax.set_title(path_label, fontsize=18, color="#3E3A36")
        ax.set_xlabel("Difficulty", fontsize=16, color="#4D463F")
        ax.set_ylabel("Accuracy", fontsize=16, color="#4D463F")
        ax.tick_params(labelsize=13, colors="#3E3A36")
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines["left"].set_color("#B8AD98")
        ax.spines["bottom"].set_color("#B8AD98")
        ax.set_ylim(0, 1)

    fig.legend(
        legend_handles,
        legend_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.06),
        ncol=4,
        frameon=True,
        framealpha=0.95,
        facecolor="#F3EFE6",
        edgecolor="#CFC6B4",
        fontsize=14,
        title="Model",
        title_fontsize=18,
    )

    plt.tight_layout(rect=[0, 0.12, 1, 1])
    plt.savefig(f"../figures/diff-results-{metric}.pdf", dpi=360, bbox_inches="tight", facecolor="white")
    plt.close()


In [58]:
results = read_json("results.json")
sub_results = [
    result for result in results
    if result["model_name"] in [
        "Llama-3.3-70B-Instruct",
        "Qwen3.5-397B-A17B",
        "GPT-4.1",
        "Qwen3-235B-A22B-Thinking",
        "DeepSeek-V3.2",
        "GPT-5.4",
        "Gemini-3-Flash",
        "Gemini-3.1-Pro",
    ] and result["thinking"] in [None, "medium", "adaptive"]
]

plot_diff_results_by_path(sub_results, path="en_path")
plot_diff_results_by_path(sub_results, path="ru_path")

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_76789/1966031826.py:88: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(
/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_76789/1966031826.py:88: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(
/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_76789/1966031826.py:88: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(
/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_76789/1966031826.py:88: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(


In [ ]:
results = read_json("results.json")
sub_results = [
    result for result in results
    if result["model_name"] in [
        "Llama-3.3-70B-Instruct",
        "Mistral-Large-3-675B-Instruct",
        "GPT-4.1",
        "Qwen3-235B-A22B-Thinking",
        "DeepSeek-V3.2",
        "GPT-5.4",
        "Gemini-3.1-Pro",
    ] and result["thinking"] in [None, "medium", "adaptive", "high"]
]

plot_diff_results_by_metric(sub_results, metric="exact_match")
plot_diff_results_by_metric(sub_results, metric="gpt-4o_judge_match")

In [72]:
# print model performance by difficulty for each model
for result in results:
    model_name = result["model_name"]
    thinking = result.get("thinking")
    key = f"{model_name} ({thinking})" if thinking is not None else model_name
    outputs = read_json(result["en_path"]) if result["en_path"] else None

    if outputs is None:
        continue

    print(f"\nModel: {key}")
    for diff_idx, diff_label in enumerate(difficulties, start=1):
        diff_outputs = [
            output for output in outputs["data"]
            if benchmark_dict.get(output["id"])
            and benchmark_dict[output["id"]]["difficulty_id"] == diff_idx
            and output["id"] in creative_ids  # Filter to creative questions only
        ]

        if not diff_outputs:
            continue

        em_acc = np.mean([int(s.get("exact_match", 0)) for s in diff_outputs])
        judge_acc = np.mean([int(s.get("gpt-4o_judge_match", 0)) for s in diff_outputs])
        print(f"  Difficulty: {diff_label}, EM Accuracy: {em_acc:.2%}, GPT-4o Judge Accuracy: {judge_acc:.2%}")


Model: OLMo-2-32B-Instruct
  Difficulty: Very Simple, EM Accuracy: 2.88%, GPT-4o Judge Accuracy: 6.95%
  Difficulty: Simple, EM Accuracy: 0.88%, GPT-4o Judge Accuracy: 4.42%
  Difficulty: Medium, EM Accuracy: 2.75%, GPT-4o Judge Accuracy: 6.42%
  Difficulty: Hard, EM Accuracy: 2.16%, GPT-4o Judge Accuracy: 4.85%
  Difficulty: Very Hard, EM Accuracy: 1.82%, GPT-4o Judge Accuracy: 3.65%

Model: C4AI-command-a
  Difficulty: Very Simple, EM Accuracy: 5.28%, GPT-4o Judge Accuracy: 12.95%
  Difficulty: Simple, EM Accuracy: 4.86%, GPT-4o Judge Accuracy: 9.27%
  Difficulty: Medium, EM Accuracy: 3.90%, GPT-4o Judge Accuracy: 8.72%
  Difficulty: Hard, EM Accuracy: 4.31%, GPT-4o Judge Accuracy: 8.36%
  Difficulty: Very Hard, EM Accuracy: 3.91%, GPT-4o Judge Accuracy: 5.73%

Model: Llama-3.3-70B-Instruct
  Difficulty: Very Simple, EM Accuracy: 9.11%, GPT-4o Judge Accuracy: 16.07%
  Difficulty: Simple, EM Accuracy: 4.42%, GPT-4o Judge Accuracy: 10.38%
  Difficulty: Medium, EM Accuracy: 4.36%, GPT-